# Real Crypto Carry IBKR Colab

This notebook is real-data-only. It does not download synthetic or proxy market data. Upload real `cme_curve.csv` and `long_prices.csv`, then build artifacts.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/real_crypto_carry_ibkr'
Path(DRIVE_ROOT, 'data').mkdir(parents=True, exist_ok=True)
Path(DRIVE_ROOT, 'artifacts', 'latest').mkdir(parents=True, exist_ok=True)

In [ ]:
# Run this from a cloned copy of the repo, or upload the repo folder to Colab first.
import os, subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'numpy', 'pyyaml', 'ib_insync'])
repo = '/content/real_crypto_carry_ibkr'
if os.path.exists(repo):
    sys.path.insert(0, repo + '/src')
else:
    print('Clone the repo to /content/real_crypto_carry_ibkr before running artifact build cells.')

In [ ]:
CURVE_CSV = DRIVE_ROOT + '/data/cme_curve.csv'
LONG_PRICES_CSV = DRIVE_ROOT + '/data/long_prices.csv'
OUT_DIR = DRIVE_ROOT + '/artifacts/latest'
DATA_SOURCE = 'databento'  # change only to another accepted real source from config/default.yaml

print('Curve:', CURVE_CSV)
print('Long prices:', LONG_PRICES_CSV)
print('Out:', OUT_DIR)

In [ ]:
import json
from pathlib import Path
from real_crypto_carry_ibkr.artifacts import write_artifacts
from real_crypto_carry_ibkr.config import load_config
from real_crypto_carry_ibkr.data import build_provenance, load_curve, load_long_prices
from real_crypto_carry_ibkr.strategy import run_research

cfg = load_config(repo + '/config/default.yaml')
provenance = build_provenance(CURVE_CSV, LONG_PRICES_CSV, DATA_SOURCE, cfg)
curve = load_curve(CURVE_CSV)
prices = load_long_prices(LONG_PRICES_CSV)
research = run_research(curve, prices, cfg)
zip_path = write_artifacts(OUT_DIR, cfg, provenance, research, CURVE_CSV, LONG_PRICES_CSV)
print('Artifact zip:', zip_path)
print(Path(OUT_DIR, 'status.json').read_text())